In [2]:
from ultralytics import YOLO

# 1. Load a pretrained model (YOLOv8 Nano is great for small objects like checkboxes)
model = YOLO("yolov8n.pt") 
model

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/Users/gauurab/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [2]:
"""
Converts your JSON ground-truth labels → YOLO .txt format.

JSON input per file:
  { "checkboxes": [[cx,cy], ...],       ← class 0
    "lines":      [[x1,y1,x2,y2], ...], ← class 1
    "boxes":      [[x1,y1,x2,y2], ...]} ← class 2

YOLO output per line:
  <class_id> <cx_norm> <cy_norm> <w_norm> <h_norm>
"""

import os, json
from PIL import Image

# ── CONFIG ──────────────────────────────────────────
GT_FOLDER       = "clicking_mechanism/annotations"
IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels" 
LINE_PADDING    = 0.01   # 1% padding on each side of a line bbox
CHECKBOX_SIZE   = 0.02   # checkboxes get a 2% fixed-size bbox
# ────────────────────────────────────────────────────

def point_to_yolo(cx, cy, w, h):
    return cx/w, cy/h, CHECKBOX_SIZE, CHECKBOX_SIZE

def line_to_yolo(x1, y1, x2, y2, w, h):
    xmin = max(0,  min(x1,x2) - LINE_PADDING * w)
    xmax = min(w,  max(x1,x2) + LINE_PADDING * w)
    ymin = max(0,  min(y1,y2) - LINE_PADDING * h)
    ymax = min(h,  max(y1,y2) + LINE_PADDING * h)
    return (xmin+xmax)/(2*w), (ymin+ymax)/(2*h), (xmax-xmin)/w, (ymax-ymin)/h

def box_to_yolo(x1, y1, x2, y2, w, h):
    return (x1+x2)/(2*w), (y1+y2)/(2*h), abs(x2-x1)/w, abs(y2-y1)/h

def convert():
    os.makedirs(YOLO_LABELS_DIR, exist_ok=True)
    # Only process files present in BOTH folders
    gt_files    = set(f for f in os.listdir(GT_FOLDER)    if f.endswith(".json"))
    img_files   = set(f.replace(".png",".json") for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png"))
    files       = gt_files & img_files
    skipped     = gt_files - img_files

    for fname in skipped:
        print(f"[SKIP] No image for {fname}")

    for fname in files:
        img_w, img_h = Image.open(os.path.join(IMAGE_FOLDER, fname.replace(".json",".png"))).size
        data = json.load(open(os.path.join(GT_FOLDER, fname)))
        rows = []

        for cx, cy           in data.get("checkboxes", []):
            rows.append("0 {:.6f} {:.6f} {:.6f} {:.6f}".format(*point_to_yolo(cx, cy, img_w, img_h)))
        for x1,y1,x2,y2      in data.get("lines", []):
            rows.append("1 {:.6f} {:.6f} {:.6f} {:.6f}".format(*line_to_yolo(x1,y1,x2,y2,img_w,img_h)))
        for x1,y1,x2,y2      in data.get("boxes", []):
            rows.append("2 {:.6f} {:.6f} {:.6f} {:.6f}".format(*box_to_yolo(x1,y1,x2,y2,img_w,img_h)))

        with open(os.path.join(YOLO_LABELS_DIR, fname.replace(".json",".txt")), "w") as f:
            f.write("\n".join(rows))

    print(f"\n✅ Converted {len(files)} files | Skipped {len(skipped)} (no image)")

if __name__ == "__main__":
    convert()


✅ Converted 407 files | Skipped 0 (no image)


In [4]:
"""
Splits data into train/val, writes dataset.yaml, and trains YOLOv8.
Run AFTER step1_convert_labels.py.
"""

import os, shutil, random, yaml
from ultralytics import YOLO

IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels" 
# ── CONFIG ──────────────────────────────────────────
DATASET_DIR     = "dataset"          # will be created
MODEL_SIZE      = "yolov8s.pt"       # n/s/m/l/x — s is a good default
EPOCHS          = 100
IMGSZ           = 640
BATCH           = 16
VAL_SPLIT       = 0.2
SEED            = 42
# ────────────────────────────────────────────────────

CLASSES = ["checkbox", "line", "box"]

def prepare_dataset():
    for split in ("train", "val"):
        os.makedirs(f"{DATASET_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{DATASET_DIR}/labels/{split}", exist_ok=True)

    # Only include images that have a matching label file
    label_stems = set(f[:-4] for f in os.listdir(YOLO_LABELS_DIR) if f.endswith(".txt"))
    image_files = [f for f in os.listdir(IMAGE_FOLDER)
                   if f.endswith(".png") and f[:-4] in label_stems]

    random.seed(SEED)
    random.shuffle(image_files)
    split_idx  = int(len(image_files) * (1 - VAL_SPLIT))
    train_imgs = image_files[:split_idx]
    val_imgs   = image_files[split_idx:]

    for split, imgs in [("train", train_imgs), ("val", val_imgs)]:
        for img_fname in imgs:
            stem = img_fname[:-4]
            shutil.copy(os.path.join(IMAGE_FOLDER,    img_fname),
                        f"{DATASET_DIR}/images/{split}/{img_fname}")
            shutil.copy(os.path.join(YOLO_LABELS_DIR, stem + ".txt"),
                        f"{DATASET_DIR}/labels/{split}/{stem}.txt")

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

    # Write dataset.yaml
    yaml_path = os.path.join(DATASET_DIR, "dataset.yaml")
    yaml_data = {
        "path": os.path.abspath(DATASET_DIR),
        "train": "images/train",
        "val":   "images/val",
        "nc":    len(CLASSES),
        "names": CLASSES,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=False)
    print(f"Dataset YAML written to {yaml_path}")
    return yaml_path

def train(yaml_path):
    model = YOLO(MODEL_SIZE)
    model.train(
        data    = yaml_path,
        epochs  = EPOCHS,
        imgsz   = IMGSZ,
        batch   = BATCH,
        project = "runs/train",
        name    = "form_detector",
    )
    print("\n✅ Training complete. Best weights: runs/train/form_detector/weights/best.pt")

if __name__ == "__main__":
    yaml_path = prepare_dataset()
    train(yaml_path)

Train: 325 | Val: 82
Dataset YAML written to dataset/dataset.yaml
New https://pypi.org/project/ultralytics/8.4.19 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.12.5 torch-2.9.1 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


      1/100         0G      3.638      4.533      2.503       1185        640: 0% ──────────── 0/21  19.1s

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile


      1/100         0G      3.608      4.441      2.376       1271        640: 5% ╸─────────── 1/21 63.0s/it 38.0s<21:01

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 33% ━━━━──────── 1/3 23.3s/it 7.0s<46.6s

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illumina

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 67% ━━━━━━━━──── 2/3 12.5s/it 13.0s<12.5s

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 5.6s/it 16.8s
                   all         82       3475      0.667      0.636      0.629      0.335
              checkbox         69       2234      0.557      0.449      0.395      0.119
                  line         55        560      0.779       0.67      0.707      0.366
                   box         54        681      0.665       0.79      0.784      0.519
Speed: 0.6ms preprocess, 177.5ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /Users/gauurab/Documents/Projects/salary-nku/camp-norse/Detect_Regions_in_PDF/runs/detect/runs/train/form_detector

✅ Training complete. Best weights: runs/train/form_detector/weights/best.pt


In [8]:
"""
Runs YOLOv8 inference on test images and computes
Precision / Recall / F1 / Accuracy (Jaccard) per class,
matching your original tolerance-based evaluation logic.
"""

import os, json, math
from PIL import Image
from ultralytics import YOLO

# ── CONFIG ──────────────────────────────────────────
WEIGHTS_PATH  = "runs/detect/runs/train/form_detector/weights/best.pt"
GT_FOLDER       = "clicking_mechanism/annotations"
IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels" 
TOLERANCES    = [0.05, 0.10, 0.20]
CONF_THRESH   = 0.25
CLASS_NAMES   = {0: "Checkboxes", 1: "Lines", 2: "Boxes"}
# ────────────────────────────────────────────────────

def load_gt(path, img_w, img_h):
    """Load GT JSON and convert everything to pixel (x1,y1,x2,y2) boxes."""
    data = json.load(open(path))
    gt = {0: [], 1: [], 2: []}

    for cx, cy in data.get("checkboxes", []):
        s = 0.01 * min(img_w, img_h)
        gt[0].append((cx-s, cy-s, cx+s, cy+s))

    for x1,y1,x2,y2 in data.get("lines", []):
        pad_x = 0.01 * img_w
        pad_y = 0.01 * img_h
        gt[1].append((min(x1,x2)-pad_x, min(y1,y2)-pad_y,
                      max(x1,x2)+pad_x, max(y1,y2)+pad_y))

    for x1,y1,x2,y2 in data.get("boxes", []):
        gt[2].append((x1, y1, x2, y2))

    return gt

def match_boxes(gt_boxes, pred_boxes, tol_x, tol_y):
    """Same order-independent closest-first matching as your original code."""
    candidates = []
    for gi, (gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
        for pi, (px1,py1,px2,py2) in enumerate(pred_boxes):
            if (abs(px1-gx1)<=tol_x and abs(py1-gy1)<=tol_y and
                abs(px2-gx2)<=tol_x and abs(py2-gy2)<=tol_y):
                dist = math.sqrt((gx1-px1)**2+(gy1-py1)**2) + \
                       math.sqrt((gx2-px2)**2+(gy2-py2)**2)
                candidates.append((dist, gi, pi))

    candidates.sort(key=lambda x: x[0])
    matched_gt, matched_pred, count = set(), set(), 0
    for _, gi, pi in candidates:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi); matched_pred.add(pi); count += 1
    return count

def compute_metrics(matches, total_pred, total_gt):
    p   = matches / total_pred if total_pred else 0
    r   = matches / total_gt   if total_gt   else 0
    f1  = (2*p*r / (p+r))      if (p+r)      else 0
    den = total_gt + total_pred - matches
    acc = matches / den         if den        else 0
    return p, r, f1, acc

def yolo_box_to_pixels(box, img_w, img_h):
    """Convert YOLO result xyxy (already in pixels) to tuple."""
    x1, y1, x2, y2 = box
    return float(x1), float(y1), float(x2), float(y2)

def evaluate():
    model = YOLO(WEIGHTS_PATH)

    # Verified file pairs only
    gt_stems  = set(f[:-5] for f in os.listdir(GT_FOLDER)    if f.endswith(".json"))
    img_stems = set(f[:-4]  for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png"))
    stems     = sorted(gt_stems & img_stems)

    if not stems:
        print("No matching GT+image pairs found. Check your paths.")
        return

    for tol in TOLERANCES:
        totals = {cls: [0,0,0] for cls in CLASS_NAMES}  # [gt, pred, match]

        print("\n" + "="*60)
        print(f"TOLERANCE: {int(tol*100)}%")
        print("="*60)

        for stem in stems:
            img_path = os.path.join(IMAGE_FOLDER, stem + ".png")
            gt_path  = os.path.join(GT_FOLDER,    stem + ".json")
            img_w, img_h = Image.open(img_path).size
            tol_x, tol_y = img_w * tol, img_h * tol

            gt = load_gt(gt_path, img_w, img_h)

            results = model.predict(img_path, conf=CONF_THRESH, verbose=False)[0]

            # Group predictions by class
            pred = {0: [], 1: [], 2: []}
            for box in results.boxes:
                cls_id = int(box.cls)
                if cls_id in pred:
                    pred[cls_id].append(yolo_box_to_pixels(box.xyxy[0], img_w, img_h))

            for cls_id in CLASS_NAMES:
                matches = match_boxes(gt[cls_id], pred[cls_id], tol_x, tol_y)
                totals[cls_id][0] += len(gt[cls_id])
                totals[cls_id][1] += len(pred[cls_id])
                totals[cls_id][2] += matches

        for cls_id, name in CLASS_NAMES.items():
            gt_n, pred_n, match_n = totals[cls_id]
            p, r, f1, acc = compute_metrics(match_n, pred_n, gt_n)
            print(f"\n{name}:")
            print(f"  Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f} | Accuracy (Jaccard): {acc:.3f}")

if __name__ == "__main__":
    evaluate()


TOLERANCE: 5%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.789 | Recall: 0.794 | F1: 0.791 | Accuracy (Jaccard): 0.655

Lines:
  Precision: 0.715 | Recall: 0.824 | F1: 0.766 | Accuracy (Jaccard): 0.620

Boxes:
  Precision: 0.540 | Recall: 0.642 | F1: 0.587 | Accuracy (Jaccard): 0.415

TOLERANCE: 10%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.808 | Recall: 0.813 | F1: 0.811 | Accuracy (Jaccard): 0.682

Lines:
  Precision: 0.736 | Recall: 0.847 | F1: 0.788 | Accuracy (Jaccard): 0.650

Boxes:
  Precision: 0.574 | Recall: 0.682 | F1: 0.623 | Accuracy (Jaccard): 0.453

TOLERANCE: 20%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.820 | Recall: 0.826 | F1: 0.823 | Accuracy (Jaccard): 0.699

Lines:
  Precision: 0.755 | Recall: 0.869 | F1: 0.808 | Accuracy (Jaccard): 0.677

Boxes:
  Precision: 0.667 | Recall: 0.793 | F1: 0.724 | Accuracy (Jaccard): 0.568


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


In [11]:
"""
Predict on a single random form image and visualize results.
"""

import os, random
from PIL import Image, ImageDraw
from ultralytics import YOLO

# ── CONFIG ──────────────────────────────────────────
WEIGHTS_PATH  = "runs/detect/runs/train/form_detector/weights/best.pt"
IMAGE_FOLDER  = "data"
OUTPUT_PATH   = "prediction_sample.png"
CONF_THRESH   = 0.25
# ────────────────────────────────────────────────────

CLASS_NAMES  = {0: "checkbox", 1: "line", 2: "box"}
CLASS_COLORS = {0: "blue",     1: "green", 2: "red"}

def predict_random():
    images = [f for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png")]
    chosen = random.choice(images)
    img_path = os.path.join(IMAGE_FOLDER, chosen)
    print(f"Running prediction on: {chosen}")

    model   = YOLO(WEIGHTS_PATH)
    results = model.predict(img_path, conf=CONF_THRESH, verbose=False)[0]

    img  = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    counts = {0: 0, 1: 0, 2: 0}
    for box in results.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
        color  = CLASS_COLORS[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, max(0, y1 - 12)), f"{CLASS_NAMES[cls_id]} {conf:.2f}", fill=color)
        counts[cls_id] += 1

    img.save(OUTPUT_PATH)

    print(f"\nDetected:")
    for cls_id, name in CLASS_NAMES.items():
        print(f"  {name}: {counts[cls_id]}")
    print(f"\nSaved to: {OUTPUT_PATH}")

if __name__ == "__main__":
    predict_random()

Running prediction on: form158.png

Detected:
  checkbox: 3
  line: 1
  box: 9

Saved to: prediction_sample.png
